# 07 — Entrenamiento y evaluación final A/B/C/D

Orden: seleccionar épocas con **B5 de desarrollo** → elegir el suavizado de D con B5 → reiniciar y entrenar con todos los bloques → congelar los modelos → abrir test → comparar A/B/C/D. Cada sujeto se entrena por separado. Se conserva el `pipeline_id` del 02.

**D (protocolo v1.4)** es la configuración de C entrenada con cinco semillas predefinidas, promediada en unidades originales y suavizada con un filtro gaussiano cuyo σ se elige en B5. La primera semilla es la de C: C es el miembro 0 de D y se entrena una sola vez. A/B/C no cambian.

La ejecución real requiere V01–V14 PASS, aprobación, 05/06 completos y decisiones explícitas sobre batch, máximo final, paciencia y `min_delta`. Las 50 épocas del piloto no fijan la duración final. Ejecutar solo después de autorizar el costo experimental.

Las celdas muestran sujetos, variantes y épocas. Las ventanas nunca cruzan bloques. B5 determina únicamente la duración de cada modelo y el σ de D; no constituye una estimación independiente del desempeño tras Optuna. El desempeño final se mide en test una vez congelado todo.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY = 'https://github.com/SaulRodas/BayesianOptimizationNN_ECOG.git'
BRANCH = 'main'
PROJECT = Path('/content/tt-ecog')
PREVIOUS = None
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPOSITORY, str(PROJECT)], check=True)
else:
    remote = subprocess.check_output(['git', '-C', str(PROJECT), 'remote', 'get-url', 'origin'], text=True).strip()
    if remote != REPOSITORY:
        raise RuntimeError('El checkout existente pertenece a otro remoto; revisa /content/tt-ecog.')
    if subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True).strip():
        raise RuntimeError('El clon de Colab tiene cambios locales; no se actualiza automáticamente.')
    PREVIOUS = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip()
    subprocess.run(['git', '-C', str(PROJECT), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only', 'origin', BRANCH], check=True)
COMMIT = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip()
if 'tt_ecog' in sys.modules and PREVIOUS is not None and PREVIOUS != COMMIT:
    raise RuntimeError('El código cambió después de importarlo: reinicia el runtime y vuelve a ejecutar.')
print('Commit en Colab:', COMMIT)
print('Debe coincidir con `git rev-parse HEAD` local (haz push antes de ejecutar).')

In [ ]:
env_check = subprocess.run([sys.executable, str(PROJECT / 'scripts/check_environment.py'), '--install-missing'],
                           capture_output=True, text=True)
print(env_check.stdout + env_check.stderr)  # Colab no muestra la salida de subprocesos sin captura.
env_check.check_returncode()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT) + '[search]'], check=True)
# El .pth del modo editable solo se procesa al iniciar el intérprete; el kernel activo no lo ve.
SOURCE = PROJECT / 'src'
if str(SOURCE) not in sys.path:
    sys.path.insert(0, str(SOURCE))
import tt_ecog
import torch
if not Path(tt_ecog.__file__).resolve().is_relative_to(SOURCE.resolve()):
    raise RuntimeError(f'tt_ecog se importa desde {tt_ecog.__file__}, fuera del checkout; reinicia el runtime.')
print('tt_ecog', tt_ecog.__version__, '| torch', torch.__version__, '| CUDA:', torch.cuda.is_available())

In [ ]:
from google.colab import drive
DRIVE_ROOT = Path('/content/drive/MyDrive')
if not DRIVE_ROOT.is_dir():
    drive.mount('/content/drive')
CONFIG = PROJECT / 'configs/experiment_config.yaml'


## Configuración y requisitos de ejecución

In [ ]:
from pathlib import Path
import gc
from time import perf_counter
import numpy as np
import pandas as pd
import torch
from tt_ecog.config import load_config
from tt_ecog import preprocessing as pp
from tt_ecog.datasets import load_fold, fit_fold_scalers, full_development, inference_window_starts
from tt_ecog.prep.folds import subject_blocks
from tt_ecog.model import architecture, build_model
from tt_ecog.losses import make_loss
from tt_ecog.training import (seed_run, fold_seed, make_loaders, training_loaders, make_optimizer, train_epoch, evaluate,
    synchronize, reset_peak_memory, peak_memory)
from tt_ecog.final_evaluation import (VARIANTS, ENSEMBLE, final_policy, selected_variants, final_runs,
    ensemble_members, epoch_selection, internal_prediction, smoothing_selection, smooth_trajectory,
    predict_trajectory, score_test, combined_timing)
from tt_ecog.io import (approved_pipeline, open_final_run, read_json, atomic_json, StoreError,
    save_training_checkpoint, load_training_checkpoint, save_final_scalers, load_final_scalers,
    save_selection_prediction, keep_selected_prediction, load_selected_prediction, save_internal_validation,
    freeze_final_run, verify_final_run, read_final_test, save_test_result, verify_manifest)
from tt_ecog.reporting import json_safe, report_final_evaluation
config = load_config(CONFIG)

In [ ]:
policy = final_policy(config)
pipeline_dir = approved_pipeline(config, DRIVE_ROOT)
if not torch.cuda.is_available():
    raise RuntimeError('La ejecución experimental requiere la GPU real de Colab; CPU solo en tests sintéticos.')
device = torch.device('cuda')
print('Pipeline:', pipeline_dir.name, '| validación interna:', policy['validation_block'], '| política:', policy)

## Parámetros A/B/C/D y corrida recuperable

A: arquitectura y AdamW de referencia. B: arquitectura de referencia y LR/WD seleccionados en 05. C: arquitectura seleccionada en 06 y el mismo LR/WD de B. D: la configuración de C con las semillas de `final_evaluation.ensemble.seed_offsets`, sumadas a la semilla común; el miembro 0 es C. La segunda tabla muestra cada modelo que se entrena, con su semilla.

Se leen estudios completos, sin generar trials. Copiar el `run_id` mostrado a `paths.final_run_id` para reanudar esta corrida tras una desconexión. Un valor `null` crea una corrida nueva. Después del acceso a test, reabrir únicamente la misma corrida para completar/exportar resultados; no cambiar modelos o decisiones en función del test.

In [ ]:
plans = {}
for subject in config.data['project']['subjects']:
    plans[subject] = selected_variants(config, DRIVE_ROOT, subject)
output = open_final_run(config, DRIVE_ROOT, plans, device)
print('paths.final_run_id:', output.name)
print('Resultados:', output)
display(pd.DataFrame([{ 'subject': s, 'variant': v, **p} for s, variants in plans.items() for v, p in variants.items()]))
display(pd.DataFrame([{'subject': s, 'run': name, 'seed': run['seed'], **run['parameters']['architecture'],
                       'learning_rate': run['parameters']['learning_rate'], 'weight_decay': run['parameters']['weight_decay']}
                      for s in plans for name, run in final_runs(config, plans[s], s).items()]))

## Seleccionar la duración con el fold B5 y el σ de D

Los escaladores de este fold se ajustaron únicamente con su entrenamiento purgado. Por época se reconstruye la trayectoria B5 y se calcula `val_pearson_full`. La paciencia cuenta épocas sin una mejora mayor que `min_delta`; la duración seleccionada corresponde al mayor Pearson observado (primer empate). Ninguna etiqueta de test interviene. Cada miembro de D sigue la misma regla con su propia semilla.

Cada época guarda también su trayectoria B5 en unidades originales, antes del checkpoint, y se comprueba que reproduce su `val_pearson_full`. Al terminar se conserva solo la de la época elegida. Con las de los miembros de D se arma su promedio y se elige el σ del suavizado: mayor Pearson macro en B5, el menor σ en un empate; σ = 0 es no suavizar.

El checkpoint conserva pesos, AdamW, generadores aleatorios y curva completa. Una época interrumpida se repite desde el último checkpoint completo.

In [ ]:
if not (output / 'frozen.json').exists():
    if (output / 'test_access.json').exists():
        raise StoreError('Ya se abrió test: no se permite volver a seleccionar épocas.')
    for subject in config.data['project']['subjects']:
        fold = load_fold(config, pipeline_dir / 'folds', subject, policy['validation_block'])
        for name, run_plan in final_runs(config, plans[subject], subject).items():
            parameters, seed = run_plan['parameters'], run_plan['seed']
            folder = output / subject / name
            folder.mkdir(parents=True, exist_ok=True)
            seed_run(seed)
            train_loader, val_loader = training_loaders(config, fold, policy['batch_size'], seed, device)
            model = build_model(config, next(iter(fold.blocks.values())).x.shape[0], architecture(config, **parameters['architecture'])).to(device)
            optimizer = make_optimizer(model, parameters['learning_rate'], parameters['weight_decay'])
            run = {'pipeline_id': pipeline_dir.name, 'run_id': output.name, 'subject': subject,
                   'variant': name, 'variant_parameters': parameters, 'seed': seed, 'phase': 'epoch_selection'}
            history = []
            checkpoint = folder / 'selection.pt'
            if checkpoint.exists():
                state = load_training_checkpoint(checkpoint, model, optimizer, train_loader, pipeline_dir.name)
                if state['parameters'] != run:
                    raise StoreError('El checkpoint de selección pertenece a otra decisión.')
                history = state['history']
            decision = epoch_selection(history, policy)
            pd.DataFrame(history).to_csv(folder / 'selection.csv', index=False)
            for epoch in range(len(history) + 1, policy['max_epochs'] + 1):
                if decision['stop']:
                    break
                reset_peak_memory(device)
                synchronize(device)
                started = perf_counter()
                train_loss = train_epoch(model, optimizer, make_loss(config), train_loader, device)
                metrics = evaluate(model, make_loss(config), fold, val_loader, device, config.data['metrics']['degenerate_std_relative'])
                synchronize(device)
                row = {'epoch': epoch, 'train_loss': train_loss, 'val_loss': metrics['val_loss'],
                       'val_pearson_full': metrics['val_pearson_full'], 'elapsed_seconds': perf_counter() - started,
                       **peak_memory(device)}
                # Trayectoria B5 de esta época, guardada antes de su checkpoint: D elige su σ con ella.
                save_selection_prediction(folder, epoch, internal_prediction(config, model, fold, policy['batch_size'],
                                                                             device, row['val_pearson_full']))
                history.append(row)
                save_training_checkpoint(checkpoint, model, optimizer, epoch, run, train_loader, history)
                pd.DataFrame(history).to_csv(folder / 'selection.csv', index=False)
                decision = epoch_selection(history, policy)
                print(subject, name, row)
            atomic_json(folder / 'selection.json', decision)
            keep_selected_prediction(folder, decision['selected_epoch'])
            print('Duración elegida:', subject, name, decision)
            del model, optimizer, train_loader, val_loader
            gc.collect()
        # D: promedio de sus miembros en su época elegida; σ con mayor Pearson macro en B5.
        block = fold.blocks[fold.val_block]
        save_internal_validation(output / subject, fold.val_block, block.y_original, block.valid)
        members = [load_selected_prediction(output / subject / name)[1] for name in ensemble_members(config)]
        smoothing = smoothing_selection(config, block.y_original, block.valid, members)
        atomic_json(output / subject / ENSEMBLE / 'selection.json', smoothing)
        print(subject, ENSEMBLE, '| σ elegido:', smoothing['selected_sigma_samples'], 'muestras',
              '| Pearson B5 de cada miembro:', smoothing['member_internal_pearson'])
        display(pd.DataFrame(smoothing['sigma_table']))
        del fold, members
else:
    verify_final_run(config, output)
    print('Duraciones y σ de D ya congelados; se conservan.')

## Escaladores y reentrenamiento con todo desarrollo

Se ajusta un solo par de escaladores por sujeto con las columnas válidas de B1–B5, compartido por A/B/C y los miembros de D. La unión de muestras sirve únicamente para calcular estadísticas. Las señales y las ventanas mantienen sus bloques independientes.

Cada modelo reinicia pesos y AdamW con su semilla (la común del sujeto para A/B/C; la suya para cada miembro adicional de D) y entrena exactamente su duración seleccionada. No hay validación, scheduler ni early stopping en este reentrenamiento.

In [ ]:
if not (output / 'frozen.json').exists():
    if (output / 'test_access.json').exists():
        raise StoreError('Ya se abrió test: no se permite reentrenar.')
    for subject in config.data['project']['subjects']:
        _, arrays = subject_blocks(pipeline_dir, subject, config.data['validation']['block_names'])
        eligible = {name: a['valid'] for name, a in arrays.items()}
        feature_scaler, target_scaler, scaler_stats = fit_fold_scalers(config, arrays, eligible, 'final development')
        spec = pp.PreprocessingSpec.from_config(config, subject)
        save_final_scalers(output / subject, feature_scaler, target_scaler, spec.channels.kept)
        fold = full_development(config, subject, arrays, feature_scaler, target_scaler)
        print(subject, scaler_stats, '| ventanas independientes:', len(fold.train_windows))
        for name, run_plan in final_runs(config, plans[subject], subject).items():
            parameters, seed = run_plan['parameters'], run_plan['seed']
            folder = output / subject / name
            epochs = read_json(folder / 'selection.json')['selected_epoch']
            seed_run(seed)
            train_loader, _ = training_loaders(config, fold, policy['batch_size'], seed, device)
            model = build_model(config, next(iter(fold.blocks.values())).x.shape[0], architecture(config, **parameters['architecture'])).to(device)
            optimizer = make_optimizer(model, parameters['learning_rate'], parameters['weight_decay'])
            run = {'pipeline_id': pipeline_dir.name, 'run_id': output.name, 'subject': subject,
                   'variant': name, 'variant_parameters': parameters, 'seed': seed, 'phase': 'full_development', 'epochs': epochs}
            history = []
            checkpoint = folder / 'last.pt'
            if checkpoint.exists():
                state = load_training_checkpoint(checkpoint, model, optimizer, train_loader, pipeline_dir.name)
                if state['parameters'] != run:
                    raise StoreError('El checkpoint final pertenece a otra decisión.')
                history = state['history']
            pd.DataFrame(history).to_csv(folder / 'refit.csv', index=False)
            for epoch in range(len(history) + 1, epochs + 1):
                reset_peak_memory(device)
                synchronize(device)
                started = perf_counter()
                loss = train_epoch(model, optimizer, make_loss(config), train_loader, device)
                synchronize(device)
                history.append({'epoch': epoch, 'train_loss': loss, 'elapsed_seconds': perf_counter() - started, **peak_memory(device)})
                save_training_checkpoint(checkpoint, model, optimizer, epoch, run, train_loader, history)
                pd.DataFrame(history).to_csv(folder / 'refit.csv', index=False)
                print(subject, name, history[-1])
            del model, optimizer, train_loader
            gc.collect()
        del fold, arrays
else:
    print('Modelos ya congelados; no se reentrenan.')

## Congelar las decisiones y los modelos antes de test

Incluye a los miembros de D. El σ guardado se vuelve a calcular con las trayectorias B5 conservadas y debe coincidir; cada trayectoria debe reproducir el Pearson de su época elegida.

In [ ]:
frozen = freeze_final_run(config, output)
print('Congelado:', frozen['run_id'], '| pipeline_id:', frozen['pipeline_id'])
display(pd.DataFrame([{'subject': s, 'run': v, **d} for s, vs in frozen['decisions'].items() for v, d in vs.items()]))
display(pd.DataFrame([{'subject': s, 'variant': ENSEMBLE, 'sigma_samples': d['selected_sigma_samples'],
                       'pearson_B5': d['selected_internal_pearson']} for s, d in frozen['ensemble'].items()]))

## Test: transformación compartida e inferencia A/B/C/D

Solo esta celda resuelve `paths.test_labels_dir` (relativo a `TT_ECoG`) y abre `test_data`/`test_dg`. Las compuertas comprueban aprobación, configuración congelada e integridad. No se ajusta ningún escalador en test.

Un segmento independiente por sujeto: exclusión → CMR → FIR → Morlet y antialias por canal → guante → retardo → máscaras y ventanas. Todos los modelos reciben exactamente las mismas entradas, tiempos y cobertura. D es el promedio de sus miembros (C incluido), suavizado con el σ congelado; también se informan cada miembro y el promedio sin suavizar. El informe usa unidades originales. Si se interrumpe, solo se repite la inferencia del sujeto que no terminó con los mismos modelos congelados.

In [ ]:
for subject in config.data['project']['subjects']:
    destination = output / subject / 'test'
    if destination.exists():
        verify_final_run(config, output)
        verify_manifest(destination, 'artifact_manifest.json')
        print(subject, 'test ya guardado; se reutiliza sin abrir etiquetas.')
        continue
    raw, glove, source = read_final_test(config, DRIVE_ROOT, output, subject)
    spec = pp.PreprocessingSpec.from_config(config, subject)
    referenced = pp.common_median_reference(pp.exclude_channels(raw, spec.channels))
    filtered = pp.apply_fir(referenced, spec.fir)
    features = np.empty((filtered.shape[0], len(spec.freqs), (raw.shape[1] + spec.factor - 1) // spec.factor), dtype=np.float32)
    for channel in range(filtered.shape[0]):
        power = pp.morlet_power(filtered[channel:channel + 1], spec.sfreq, spec.freqs, spec.n_cycles, spec.zero_mean)[0]
        features[channel] = pp.decimate(power, spec.antialias, spec.factor)
    feature_valid = pp.feature_valid_mask(raw.shape[1], spec)
    inspection = pp.glove_storage_inspection(glove, source['sampling_hz_glove_storage'], spec)
    y25, knots = pp.effective_glove(glove, inspection, spec)
    targets, target_valid = pp.segment_glove(y25, knots, 0, raw.shape[1], spec)
    features, truth, xi, yi = pp.align_delay(features, targets, spec.delay_samples)
    valid = feature_valid[xi] & target_valid[yi]
    if not valid.any() or not np.isfinite(features).all():
        raise ValueError('Test sin cobertura válida o con features no finitas.')
    starts = inference_window_starts(valid, config.data['validation']['inference_window_samples'], config.data['validation']['inference_stride_samples'])
    feature_scaler, target_scaler = load_final_scalers(config, output / subject)
    x = pp.matrix_to_features(feature_scaler.transform(pp.features_to_matrix(features)), features.shape[0], features.shape[1]).astype(np.float32)
    print(subject, '| features:', x.shape, '| tiempos válidos:', valid.sum(), '| ventanas:', len(starts))
    del raw, glove, referenced, filtered, features, power
    saved = {'truth': truth, 'valid': valid, 'x_index': xi, 'y_index': yi, 'window_starts': starts}
    runs = final_runs(config, frozen['plans'][subject], subject)
    predictions, scores, common_coverage = {}, {}, None
    for name, run_plan in runs.items():
        model = build_model(config, x.shape[0], architecture(config, **run_plan['parameters']['architecture'])).to(device)
        state = torch.load(output / subject / name / 'last.pt', map_location='cpu', weights_only=True)
        model.load_state_dict(state['model'])
        del state
        predictions[name], coverage, timing = predict_trajectory(model, x, starts, valid, target_scaler, policy['batch_size'], device)
        if common_coverage is not None and not np.array_equal(coverage, common_coverage):
            raise ValueError('Los modelos no tienen la misma cobertura.')
        common_coverage = coverage
        scores[name] = {**score_test(truth, predictions[name], valid, config), **timing}
        saved['prediction_' + name.replace('/', '_')] = predictions[name]
        print(subject, name, 'Pearson:', scores[name]['pearson_macro'])
        del model
        gc.collect()
    results = {variant: scores[variant] for variant in VARIANTS}
    # D: promedio de sus miembros (C es el miembro 0), suavizado con el σ congelado con B5.
    members, smoothing = ensemble_members(config), frozen['ensemble'][subject]
    mean = np.mean([predictions[name] for name in members], axis=0)
    prediction = smooth_trajectory(mean, valid, smoothing['selected_sigma_samples'], smoothing['truncate_sigmas'])
    results[ENSEMBLE] = {**score_test(truth, prediction, valid, config), **combined_timing([scores[n] for n in members])}
    saved['prediction_' + ENSEMBLE], saved['prediction_ensemble_mean_D'] = prediction, mean
    ensemble_report = {'sigma_samples': smoothing['selected_sigma_samples'],
                       'members': {name: {**scores[name], 'seed': runs[name]['seed']} for name in members},
                       'mean_before_smoothing': score_test(truth, mean, valid, config)}
    print(subject, ENSEMBLE, 'Pearson:', results[ENSEMBLE]['pearson_macro'], '| σ:', smoothing['selected_sigma_samples'],
          '| promedio sin suavizar:', ensemble_report['mean_before_smoothing']['pearson_macro'])
    saved['coverage'] = common_coverage
    save_test_result(destination, saved, json_safe({'pipeline_id': frozen['pipeline_id'], 'run_id': frozen['run_id'],
        'subject': subject, 'source': source, 'glove_storage': inspection, 'variants': results,
        'ensemble': ensemble_report, 'coverage_min_valid': int(common_coverage[valid].min()), 'original_units': True}))
    del x, saved, predictions, mean, prediction

## Tablas, trayectorias y cierre

El reporte se reconstruye con los resultados guardados, sin nuevas lecturas de test. Pearson macro es la métrica principal; CCC, MSE y RMSE son secundarios. La comparación principal es A/B/C/D; la tabla de D muestra cada semilla (C es el miembro 0) y su promedio sin suavizar, sin elegir ninguna por su resultado. No usar estas cifras para cambiar hiperparámetros, canales, épocas, σ o máscaras.

In [ ]:
table, comparison, ensemble_detail = report_final_evaluation(config, output)
display(table)
display(comparison)
display(ensemble_detail)
from IPython.display import Image
for subject in config.data['project']['subjects']:
    display(Image(filename=str(output / f'{subject}_predictions.png')))
print('Resultados completos:', output)